# NB_bishop_ch15_figures

**Bishop Chapter 15 — Key Figures: Mixture Models, EM Algorithm, K-Means Clustering**

This notebook generates pedagogical matplotlib figures for Bishop Chapter 15 on discrete latent variables.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_15/NB_bishop_ch15_figures.ipynb)

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import norm, multivariate_normal
from scipy.spatial import Voronoi, voronoi_plot_2d
import os

# ── colour palette ──────────────────────────────────────────────
COLORS = {
    'blue':  '#1A3A6E',
    'red':   '#CD0000',
    'green': '#2E7D32',
    'amber': '#B5853F',
}

# ── global rcParams ─────────────────────────────────────────────
matplotlib.rcParams['figure.facecolor']   = 'none'
matplotlib.rcParams['axes.facecolor']     = 'none'
matplotlib.rcParams['savefig.facecolor']  = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid']          = False
matplotlib.rcParams['legend.loc']         = 'upper center'
matplotlib.rcParams['legend.framealpha']  = 0.0

CHART_DIR = '../../charts'
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(fig, name, chart_dir=CHART_DIR):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

print('Setup complete.')

## Figure 15.1 — Gaussian Mixture Model: Individual Components and Mixture

In [ ]:
np.random.seed(42)

x = np.linspace(-6, 10, 500)

# Three Gaussian components
means  = [-1.0, 3.0, 7.0]
stds   = [0.8, 1.5, 0.6]
weights = [0.3, 0.5, 0.2]

comp_colors = [COLORS['blue'], COLORS['red'], COLORS['green']]
comp_labels = [
    f'$\\pi_1={weights[0]}$, $\\mu_1={means[0]}$, $\\sigma_1={stds[0]}$',
    f'$\\pi_2={weights[1]}$, $\\mu_2={means[1]}$, $\\sigma_2={stds[1]}$',
    f'$\\pi_3={weights[2]}$, $\\mu_3={means[2]}$, $\\sigma_3={stds[2]}$',
]

fig, ax = plt.subplots(figsize=(9, 5))

mixture = np.zeros_like(x)
for mu, sigma, pi, col, lab in zip(means, stds, weights, comp_colors, comp_labels):
    component = pi * norm.pdf(x, mu, sigma)
    mixture += component
    ax.plot(x, component, '--', color=col, lw=1.5, alpha=0.7, label=lab)
    ax.fill_between(x, component, alpha=0.06, color=col)

ax.plot(x, mixture, color='k', lw=2.5, label='Mixture $p(x)$')

ax.set_xlabel('$x$')
ax.set_ylabel('$p(x)$')
ax.set_title('Gaussian Mixture Model ($K=3$)')
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig15_1_mixture_components')
plt.show()

## Figure 15.3 — EM Algorithm Visualization (E-step and M-step)

In [ ]:
np.random.seed(10)

# Generate 2D data from 3 Gaussians
true_means = [np.array([-2, -1]), np.array([2, 2]), np.array([0, -3])]
true_covs  = [0.5 * np.eye(2), 0.8 * np.eye(2), 0.4 * np.eye(2)]
true_ns    = [60, 80, 50]

data = np.vstack([
    np.random.multivariate_normal(m, c, n)
    for m, c, n in zip(true_means, true_covs, true_ns)
])
N = len(data)
K = 3
comp_colors = [COLORS['blue'], COLORS['red'], COLORS['green']]

# ── EM algorithm with history ────────────────────────────────
# Random initialization
mu = np.array([[-3, 1], [3, -2], [1, 1]], dtype=float)
sigma = [1.0 * np.eye(2) for _ in range(K)]
pi_k = np.ones(K) / K
resp = np.zeros((N, K))

history = []  # store (mu, sigma, resp, title)

# Record initial state
history.append((mu.copy(), [s.copy() for s in sigma],
                np.ones((N, K)) / K, 'Initialization'))

for it in range(15):
    # E-step: compute responsibilities
    for k in range(K):
        resp[:, k] = pi_k[k] * multivariate_normal.pdf(data, mu[k], sigma[k])
    resp /= resp.sum(axis=1, keepdims=True)

    if it == 0:
        history.append((mu.copy(), [s.copy() for s in sigma],
                        resp.copy(), 'After E-step 1'))

    # M-step: update parameters
    N_k = resp.sum(axis=0)
    for k in range(K):
        mu[k] = (resp[:, k, None] * data).sum(axis=0) / N_k[k]
        diff = data - mu[k]
        sigma[k] = (resp[:, k, None, None] * (diff[:, :, None] * diff[:, None, :])).sum(axis=0) / N_k[k]
    pi_k = N_k / N

    if it == 0:
        history.append((mu.copy(), [s.copy() for s in sigma],
                        resp.copy(), 'After M-step 1'))

# Final state
history.append((mu.copy(), [s.copy() for s in sigma],
                resp.copy(), 'Converged'))

# ── Plot 4 panels ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (h_mu, h_sigma, h_resp, h_title) in zip(axes.flat, history):
    # Color each point by max responsibility
    labels = np.argmax(h_resp, axis=1)

    for k in range(K):
        mask = labels == k
        if mask.any():
            ax.scatter(data[mask, 0], data[mask, 1], c=comp_colors[k],
                       alpha=0.5, s=15, edgecolors='none')

    # Draw component means and covariance ellipses
    for k in range(K):
        ax.plot(h_mu[k, 0], h_mu[k, 1], 'X', color=comp_colors[k],
                ms=14, markeredgecolor='k', markeredgewidth=1.2, zorder=10)

        # Covariance ellipse (2-sigma)
        eigvals, eigvecs = np.linalg.eigh(h_sigma[k])
        angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
        width, height = 2 * 2 * np.sqrt(np.maximum(eigvals, 1e-6))
        ell = matplotlib.patches.Ellipse(
            h_mu[k], width, height, angle=angle,
            fill=False, edgecolor=comp_colors[k], lw=2, ls='--')
        ax.add_patch(ell)

    ax.set_title(h_title, fontsize=12)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_xlim(-5.5, 5.5)
    ax.set_ylim(-5.5, 5.5)

fig.suptitle('EM Algorithm for Gaussian Mixture Model', fontsize=14, y=1.01)
fig.tight_layout()
save_fig(fig, 'fig15_3_em_algorithm')
plt.show()

## Figure 15.5 — K-Means Clustering with Voronoi Boundaries

In [ ]:
np.random.seed(5)

# Generate 2D data from 3 clusters
cluster_means = [np.array([-3, 2]), np.array([3, 3]), np.array([1, -2])]
cluster_covs  = [0.6 * np.eye(2), 0.5 * np.eye(2), 0.7 * np.eye(2)]
cluster_ns    = [80, 70, 90]

X = np.vstack([
    np.random.multivariate_normal(m, c, n)
    for m, c, n in zip(cluster_means, cluster_covs, cluster_ns)
])

# Run K-means
K = 3
centroids = X[np.random.choice(len(X), K, replace=False)]

for _ in range(20):
    # Assign to nearest centroid
    dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
    labels = np.argmin(dists, axis=1)
    # Update centroids
    new_centroids = np.array([X[labels == k].mean(axis=0) for k in range(K)])
    if np.allclose(centroids, new_centroids):
        break
    centroids = new_centroids

fig, ax = plt.subplots(figsize=(8, 6))

# Voronoi boundaries
# Add far-away points to bound the Voronoi diagram
far = 50
augmented = np.vstack([centroids,
                       [far, far], [-far, far], [far, -far], [-far, -far]])
vor = Voronoi(augmented)

# Draw Voronoi edges (only from real centroids)
for ridge_idx, (p1, p2) in enumerate(vor.ridge_vertices):
    if p1 >= 0 and p2 >= 0:
        v1, v2 = vor.vertices[p1], vor.vertices[p2]
        ax.plot([v1[0], v2[0]], [v1[1], v2[1]], 'k--', lw=1.5, alpha=0.5)

comp_colors_list = [COLORS['blue'], COLORS['red'], COLORS['green']]
for k in range(K):
    mask = labels == k
    ax.scatter(X[mask, 0], X[mask, 1], c=comp_colors_list[k], s=20,
               alpha=0.5, edgecolors='none', label=f'Cluster {k+1}')

# Centroids
for k in range(K):
    ax.plot(centroids[k, 0], centroids[k, 1], '*', color=comp_colors_list[k],
            ms=20, markeredgecolor='k', markeredgewidth=1.2, zorder=10)

ax.set_xlim(-6, 6)
ax.set_ylim(-5, 6)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('K-Means Clustering ($K=3$) with Voronoi Boundaries')
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig15_5_kmeans')
plt.show()

## Figure 15.7 — EM Convergence: Log-Likelihood vs Iteration

In [ ]:
np.random.seed(10)

# Reuse data from fig15_3
true_means_conv = [np.array([-2, -1]), np.array([2, 2]), np.array([0, -3])]
true_covs_conv  = [0.5 * np.eye(2), 0.8 * np.eye(2), 0.4 * np.eye(2)]
true_ns_conv    = [60, 80, 50]

data_conv = np.vstack([
    np.random.multivariate_normal(m, c, n)
    for m, c, n in zip(true_means_conv, true_covs_conv, true_ns_conv)
])
N_conv = len(data_conv)
K_conv = 3

# Initialize
mu_c = np.array([[-3, 1], [3, -2], [1, 1]], dtype=float)
sigma_c = [1.0 * np.eye(2) for _ in range(K_conv)]
pi_c = np.ones(K_conv) / K_conv
resp_c = np.zeros((N_conv, K_conv))

log_likes = []

for it in range(30):
    # E-step
    for k in range(K_conv):
        resp_c[:, k] = pi_c[k] * multivariate_normal.pdf(data_conv, mu_c[k], sigma_c[k])
    resp_sum = resp_c.sum(axis=1, keepdims=True)
    resp_c /= resp_sum

    # Log-likelihood
    ll = np.log(resp_sum.squeeze()).sum()
    log_likes.append(ll)

    # M-step
    N_k_c = resp_c.sum(axis=0)
    for k in range(K_conv):
        mu_c[k] = (resp_c[:, k, None] * data_conv).sum(axis=0) / N_k_c[k]
        diff = data_conv - mu_c[k]
        sigma_c[k] = (resp_c[:, k, None, None] * (diff[:, :, None] * diff[:, None, :])).sum(axis=0) / N_k_c[k]
    pi_c = N_k_c / N_conv

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(range(1, len(log_likes) + 1), log_likes, 'o-',
        color=COLORS['blue'], lw=2, ms=5, markeredgecolor='k', markeredgewidth=0.5)

ax.set_xlabel('EM Iteration')
ax.set_ylabel('Log-Likelihood $\\mathcal{L}(\\theta)$')
ax.set_title('EM Convergence: Log-Likelihood is Monotonically Non-Decreasing')

# Annotate convergence region
conv_iter = 8
ax.axvline(conv_iter, ls=':', color=COLORS['green'], lw=1.5)
ax.annotate('Converged', xy=(conv_iter + 0.5, log_likes[conv_iter - 1]),
            fontsize=10, color=COLORS['green'], style='italic')

fig.tight_layout()
save_fig(fig, 'fig15_7_em_convergence')
plt.show()

In [ ]:
print('All Chapter 15 figures generated.')